# 20_XGBoost

In [1]:
import pandas as pd
from sklearn.datasets import load_iris, load_wine, load_breast_cancer
from sklearn.datasets import fetch_california_housing
from google.colab import drive
import os

# Google Drive Mount
drive.mount('/content/drive')

# Dataset을 저장할 Folder
data_path = "/content/drive/My Drive/Colab Notebooks/0_ML/Input/"

Mounted at /content/drive


# XGBoost

Gradient Boosting은 이전 Model의 Error를 줄이는 새로운 Weak Learner를 순차적으로 추가한다.

XGBoost는 이러한 Gradient Boosting의 아이디어를 기반으로 만들어진 강력한 Boosting Method이다.

XGBoost는 다음을 의미한다.

> eXtreme Gradient Boosting

기본적인 아이디어는 Gradient Boosting과 같다.

> Current Model → Error → New Tree → Improved Model → Repeat

그러나 XGBoost는 Training 과정과 Model의 복잡도를 보다 효과적으로 제어할 수 있도록 여러 기능을 추가하였다.

## From Gradient Boosting to XGBoost

Gradient Boosting과 XGBoost는 서로 완전히 다른 아이디어의 Algorithm이 아니다.

둘 다 새로운 Tree를 순차적으로 추가하면서 현재 Model의 Error를 줄여 나간다.

Gradient Boosting

> Sequential Trees → Reduce Prediction Error

XGBoost

> Gradient Boosting + Regularization + Efficient Training + Additional Controls

따라서 XGBoost는 Gradient Boosting을 보다 강력하고 실용적으로 구현한 방법으로 이해할 수 있다.

## Controlling Model Complexity

Boosting에서 Tree를 계속 추가하면 Training Data에 지나치게 맞는 복잡한 Model이 만들어질 수 있다.

XGBoost는 Model의 복잡도를 조절할 수 있는 여러 Parameters를 제공한다.

대표적인 Parameters는 다음과 같다.

- `n_estimators`: Number of Trees
- `learning_rate`: 각 Tree의 영향
- `max_depth`: 각 Tree의 최대 깊이
- `subsample`: 각 Tree Training에 사용할 Samples의 비율
- `colsample_bytree`: 각 Tree Training에 사용할 Features의 비율

이러한 Parameters를 이용하여 Model의 Learning과 Complexity를 조절할 수 있다.

## Gradient Boosting vs XGBoost

이번 실습에서는 동일한 Data를 이용하여
두 Methods의 공통점:

- Decision Trees를 순차적으로 추가한다.
- 새로운 Tree가 현재 Model의 Error를 줄이도록 Training한다.
- `learning_rate`를 이용하여 새로운 Tree의 영향을 조절할 수 있다.

XGBoost에서는 여기에 Model Complexity를 제어하기 위한 기능과 효율적인 Training 방법 두 Methods의 결과를 직접 비교한다.

XGBoost는 Gradient Boosting을 발전시킨 강력한 Method이지만,
항상 더 높은 Accuracy를 보장하는 것은 아니다.

Performance는

- Data
- Model Structure
- Parameter Settings

등에 따라 달라질 수 있다.

따라서 실제 Machine Learning에서는
Algorithm의 이름만으로 성능을 판단하지 않고
Data에 직접 적용하여 결과를 비교해야 한다.

## Classification Data

`xgboost_data.csv`는 여러 Features를 이용하여 두 Classes 중 하나를 Prediction하는 Classification Data이다.

먼저 Data를 Training Data와 Test Data로 나눈다.

In [9]:
import pandas as pd

from sklearn.model_selection import train_test_split

data = pd.read_csv(
    "/content/drive/My Drive/Colab Notebooks/0_ML/Input/xgboost_data.csv"
)

X = data.drop(
    columns=["y"]
)

y = data["y"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Data:", X_train.shape)
print("Test Data:", X_test.shape)

Training Data: (2400, 10)
Test Data: (600, 10)


## Gradient Boosting

먼저 이전 강의에서 사용한 Gradient Boosting을 동일한 Data에 적용한다.

In [10]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

gb.fit(
    X_train,
    y_train
)

gb_accuracy = gb.score(
    X_test,
    y_test
)

print(
    "Gradient Boosting Accuracy:",
    f"{gb_accuracy:.3f}"
)

Gradient Boosting Accuracy: 0.793


## XGBoost

이번에는 동일한 Training Data에 XGBoost를 적용한다.

`XGBClassifier`는 scikit-learn의 Classifier와 유사한 방법으로 사용할 수 있다.

In [11]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

xgb.fit(
    X_train,
    y_train
)

xgb_accuracy = xgb.score(
    X_test,
    y_test
)

print(
    "XGBoost Accuracy:",
    f"{xgb_accuracy:.3f}"
)

XGBoost Accuracy: 0.797


## Compare the Models

동일한 Data를 이용하여 Gradient Boosting과 XGBoost의 Test Accuracy를 비교한다.

높은 Accuracy만으로 어떤 Method가 항상 더 우수하다고 판단할 수는 없다.

Data와 Parameter 설정에 따라 결과는 달라질 수 있다.

In [12]:
print(
    "Gradient Boosting:",
    f"{gb_accuracy:.3f}"
)

print(
    "XGBoost:",
    f"{xgb_accuracy:.3f}"
)

Gradient Boosting: 0.793
XGBoost: 0.797


## Effect of Tree Depth

XGBoost에서는 각 Tree의 Complexity를 `max_depth`로 조절할 수 있다.

작은 `max_depth`는 단순한 Trees를 만든다.

큰 `max_depth`는 하나의 Tree가 더 복잡한 Relationship을 학습할 수 있도록 한다.

그러나 복잡한 Tree가 항상 새로운 Data에 더 좋은 Prediction을 하는 것은 아니다.

직접 확인해 보자.

In [13]:
for depth in [1, 2, 3, 5, 8]:

    model = XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=depth,
        random_state=42
    )

    model.fit(
        X_train,
        y_train
    )

    train_accuracy = model.score(
        X_train,
        y_train
    )

    test_accuracy = model.score(
        X_test,
        y_test
    )

    print(
        f"max_depth={depth}, "
        f"Train={train_accuracy:.3f}, "
        f"Test={test_accuracy:.3f}"
    )

max_depth=1, Train=0.692, Test=0.682
max_depth=2, Train=0.765, Test=0.740
max_depth=3, Train=0.860, Test=0.797
max_depth=5, Train=0.983, Test=0.838
max_depth=8, Train=1.000, Test=0.818


## Learning Rate and Number of Trees

Gradient Boosting과 마찬가지로 XGBoost에서도 `learning_rate`와 `n_estimators`는 서로 관련되어 있다.

작은 Learning Rate를 사용하면 각각의 Tree가 Model을 조금씩 수정한다.

따라서 충분한 수의 Trees가 필요할 수 있다.

반대로 큰 Learning Rate는 각각의 Tree가 Model을 더 크게 수정한다.

In [14]:
for lr in [0.01, 0.1, 0.3]:

    for n in [50, 100, 200]:

        model = XGBClassifier(
            n_estimators=n,
            learning_rate=lr,
            max_depth=3,
            random_state=42
        )

        model.fit(
            X_train,
            y_train
        )

        accuracy = model.score(
            X_test,
            y_test
        )

        print(
            f"Learning Rate={lr}, "
            f"Estimators={n}, "
            f"Accuracy={accuracy:.3f}"
        )

Learning Rate=0.01, Estimators=50, Accuracy=0.613
Learning Rate=0.01, Estimators=100, Accuracy=0.680
Learning Rate=0.01, Estimators=200, Accuracy=0.742
Learning Rate=0.1, Estimators=50, Accuracy=0.773
Learning Rate=0.1, Estimators=100, Accuracy=0.797
Learning Rate=0.1, Estimators=200, Accuracy=0.842
Learning Rate=0.3, Estimators=50, Accuracy=0.847
Learning Rate=0.3, Estimators=100, Accuracy=0.862
Learning Rate=0.3, Estimators=200, Accuracy=0.860


## Feature Importance

XGBoost에서도 Prediction에 각 Feature가 얼마나 사용되었는지를 살펴볼 수 있다.

이를 이용하여 Model이 어떤 Features를 중요하게 사용했는지 확인할 수 있다.

In [15]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": xgb.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

print(importance)

  Feature  Importance
3      x4    0.210438
0      x1    0.163548
4      x5    0.128150
1      x2    0.104202
7      x8    0.093147
2      x3    0.080092
6      x7    0.078990
5      x6    0.071855
9     x10    0.037383
8      x9    0.032194


## From One Tree to XGBoost

지금까지 학습한 Tree-based Methods를 비교하면 다음과 같다.

Decision Tree

> One Tree

Random Forest

> Many Independent Randomized Trees → Combine Predictions

AdaBoost

> Sequential Weak Learners → Focus More on Previous Errors

Gradient Boosting

> Sequential Trees → Reduce Current Model Error

XGBoost

> Gradient Boosting + Model Complexity Control + Efficient Training

## Summary

XGBoost는 Gradient Boosting의 기본 아이디어를 발전시킨 Boosting Method이다.

핵심 사항:

- Trees를 순차적으로 Training한다.
- 새로운 Tree가 현재 Model의 Error를 줄이는 방향으로 추가된다.
- `learning_rate`는 새로운 Tree의 영향을 조절한다.
- `n_estimators`는 추가되는 Trees의 수를 결정한다.
- `max_depth`는 각각의 Tree의 Complexity를 조절한다.
- Training Performance와 Test Performance를 함께 확인해야 한다.

## Homework - Problem 20

`xgboost_data2.csv`를 이용한다.

다음 조건을 사용한다.

- `test_size=0.2`
- `random_state=42`
- `n_estimators=100`
- `learning_rate=0.1`

`max_depth`를 다음과 같이 변경하여 XGBoost를 Training한다.

`1, 2, 3, 5, 8`

다음 질문에 답하시오.

- 각 `max_depth`의 Test Accuracy를 소수점 셋째 자리까지 적으시오.
- 가장 높은 Test Accuracy를 보인 `max_depth`를 모두 적으시오.

답만 제출하시오.[링크 텍스트](https://)